# Demand Forecasting
## Using Histogram Binning with Gradient Boosting Regressor

---

In [ ]:
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
from pathlib import Path


In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import combine_raw_data

PROCESSED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "combined_demand.csv"

combine_raw_data()
df = pd.read_csv(PROCESSED_CSV_PATH, parse_dates=["SETTLEMENT_DATE"])

df.head()

c:\Users\roodruh\Desktop\repos\neso-demand-forecasting\src\data_processing.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])


Successfully combined 5 files into: C:\Users\roodruh\Desktop\repos\neso-demand-forecasting\data\processed\combined_demand.csv


,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,SCOTTISH_TRANSFER
0,2021-01-01,1,28354,28969,26130,1158,6527,0,13471,0,...,-1,0,215,203,999,0,0,0,0,NaN
1,2021-01-01,2,28501,29114,26281,1208,6527,0,13471,0,...,-1,0,359,203,999,0,0,0,0,NaN
2,2021-01-01,3,27759,28376,25557,1202,6527,0,13471,0,...,-1,0,362,202,999,0,0,0,0,NaN
3,2021-01-01,4,26912,27749,24792,1226,6527,0,13471,0,...,-1,0,361,203,1000,0,0,0,0,NaN
4,2021-01-01,5,26004,27178,23933,1193,6527,0,13471,0,...,-1,0,304,203,1000,0,0,0,0,NaN


In [36]:
df["SETTLEMENT_DATE"] = pd.to_datetime(df["SETTLEMENT_DATE"])

df = df.sort_values(by = ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]).reset_index(drop=True)


df["ND_LAG_48"] = df["ND"].shift(48)  # 24 hours ago (48 half-hours)
df["ND_LAG_336"] = df["ND"].shift(336)  # 7 days ago (336 half-hours)
df["ND_AVG_7D"] = (
    df["ND"].shift(1).rolling(window=336).mean()
)

df = df.dropna().reset_index(drop=True)

In [34]:
non_feature_cols = ["ND", "SETTLEMENT_DATE"]

split_date = "2025-01-01"

train_mask = df["SETTLEMENT_DATE"] < split_date
test_mask = df["SETTLEMENT_DATE"] >= split_date

X_train = df.loc[train_mask].drop(columns=non_feature_cols)
y_train = df.loc[train_mask, "ND"]

X_test = df.loc[test_mask].drop(columns=non_feature_cols)
y_test = df.loc[test_mask, "ND"]